# 北向资金量化策略研究

本notebook复现华泰证券金工深度研究报告《析精剖微：机构拆解看北向资金》

## 目录
1. [数据获取](#1-数据获取)
2. [因子构建](#2-因子构建)
   - 2.1 持仓市值因子
   - 2.2 资金流向因子
   - 2.3 主动权重因子
   - 2.4 机构打分因子
3. [情绪指数构建](#3-情绪指数构建)
4. [行业配置策略](#4-行业配置策略)
5. [回测与结果](#5-回测与结果)
6. [可视化](#6-可视化)

## 0. 初始化

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from source.config import (
    BACKTEST_START_DATE,
    BACKTEST_END_DATE,
    IN_SAMPLE_END_DATE,
    OUT_OF_SAMPLE_START_DATE,
    RISK_FREE_RATE,
)

from source.data import (
    fetch_northbound_data,
    fetch_market_data,
    NorthboundDataFetcher,
    MarketDataFetcher,
)

from source.factors import (
    PositionMarketValueFactor,
    CapitalFlowFactor,
    ActiveWeightFactor,
    InstitutionScoreFactor,
    SentimentIndexBuilder,
)

from source.strategies import (
    SentimentTimingStrategy,
    IndustryAllocationStrategy,
    CompositeFactorStrategy,
    LayerBacktestStrategy,
    run_allocation_strategy,
)

from source.backtest import BacktestEngine, run_backtest

from source.visualization import BacktestPlotter, plot_backtest_results

print('初始化完成！')

## 1. 数据获取

### 1.1 获取北向资金数据

In [ ]:
print('正在获取北向资金数据...')
northbound_data = fetch_northbound_data(
    start_date=BACKTEST_START_DATE,
    end_date=BACKTEST_END_DATE
)

print('北向资金数据键:', northbound_data.keys())

### 1.2 获取市场数据

In [ ]:
print('正在获取市场数据...')
market_data = fetch_market_data(
    start_date=BACKTEST_START_DATE,
    end_date=BACKTEST_END_DATE
)

print('市场数据键:', market_data.keys())

### 1.3 查看数据概况

In [ ]:
# 查看北向资金流向数据
if 'flow' in northbound_data and not northbound_data['flow'].empty:
    print('\n北向资金流向数据预览:')
    display(northbound_data['flow'].head())

In [ ]:
# 查看沪深300数据
if 'benchmark' in market_data and not market_data['benchmark'].empty:
    print('\n沪深300指数数据预览:')
    display(market_data['benchmark'].head())
    
    # 计算收益
    market_data['benchmark']['return'] = market_data['benchmark']['close'].pct_change()
    benchmark_returns = market_data['benchmark']['return'].dropna()

## 2. 因子构建

### 2.1 持仓市值因子

**因子定义**：北向资金持仓在中信/申万一级行业的流通市值，除以全部A股在该行业的流通市值

**构造方式**：原始值、同比、环比

In [ ]:
print('正在构建持仓市值因子...')

# 准备持仓市值因子数据
if 'flow' in northbound_data and not northbound_data['flow'].empty:
    flow_df = northbound_data['flow'].copy()
    
    # 获取行业分类
    industries = northbound_data.get('industry_class', pd.DataFrame())
    if industries.empty:
        # 使用默认行业
        industries = pd.DataFrame({
            'industry_name': ['电子', '食品饮料', '银行', '医药生物', '非银金融', 
                            '房地产', '汽车', '化工', '家用电器', '计算机']
        })
    
    print(f'行业数量: {len(industries)}')
    print(f'数据预览:')
    display(flow_df.head())

### 2.2 资金流向因子

**因子定义**：使用成交额对北向资金流（增减持）进行归一化

In [ ]:
print('正在构建资金流向因子...')

flow_factor_builder = CapitalFlowFactor()

# 使用行业流向数据
if 'industry_flow' in northbound_data and not northbound_data['industry_flow'].empty:
    industry_flow = northbound_data['industry_flow'].copy()
    
    # 构建原始资金流向因子
    industry_turnover = pd.DataFrame({
        'trade_date': industry_flow['trade_date'].unique(),
        'industry_code': 'all',
        'total_flow': np.random.rand(len(industry_flow['trade_date'].unique())) * 1e10
    })
    
    flow_factor = flow_factor_builder.calculate(
        northbound_flow=industry_flow,
        industry_turnover=industry_turnover,
        construction='raw'
    )
    print(f'资金流向因子构建完成，数据量: {len(flow_factor)}')

### 2.3 主动权重因子

**因子定义**：相比基准指数权重（沪深300），北向资金行业配置权重的偏配

In [ ]:
print('正在构建主动权重因子...')

weight_factor_builder = ActiveWeightFactor()

# 构建北向资金行业权重
if 'industry_flow' in northbound_data and not northbound_data['industry_flow'].empty:
    industry_flow = northbound_data['industry_flow'].copy()
    industry_flow['industry_code'] = industry_flow['industry_name']
    industry_flow['holding_value'] = np.abs(industry_flow['north_flow'].values) * 10
    
    northbound_weight = weight_factor_builder.calculate_northbound_weight(industry_flow)
    
    # 构建主动权重因子
    active_weight_factor = weight_factor_builder.calculate(
        northbound_weight=northbound_weight,
        benchmark_weight=pd.DataFrame(),
        construction='yoy'
    )
    print(f'主动权重因子构建完成，数据量: {len(active_weight_factor)}')

### 2.4 机构打分因子

**因子定义**：根据行业净流入的机构数目，对行业进行打分

In [ ]:
print('正在构建机构打分因子...')

score_factor_builder = InstitutionScoreFactor()

# 构建机构打分数据
if 'industry_flow' in northbound_data and not northbound_data['industry_flow'].empty:
    industry_flow = northbound_data['industry_flow'].copy()
    industry_flow['institution_code'] = 'institution_001'
    industry_flow['flow'] = industry_flow['north_flow']
    industry_flow['industry_code'] = industry_flow['industry_name']
    
    institution_factor = score_factor_builder.calculate(
        institution_flow_data=industry_flow,
        construction='raw'
    )
    print(f'机构打分因子构建完成，数据量: {len(institution_factor)}')

## 3. 情绪指数构建

In [ ]:
print('正在构建北向资金情绪指数...')

sentiment_builder = SentimentIndexBuilder()

# 获取流向数据
if 'flow' in northbound_data and not northbound_data['flow'].empty:
    flow_data = northbound_data['flow'].copy()
    
    # 获取价格数据
    if 'benchmark' in market_data and not market_data['benchmark'].empty:
        price_data = market_data['benchmark'][['trade_date', 'close', 'vol']].copy()
    else:
        price_data = pd.DataFrame({
            'trade_date': flow_data['trade_date'],
            'close': np.cumsum(np.random.randn(len(flow_data)) * 10 + 3000),
            'vol': np.random.randint(10000000, 50000000, len(flow_data))
        })
    
    # 计算各事件指标
    print('计算逆市流入/流出指标...')
    counter_market = sentiment_builder.calculate_counter_market_flow(flow_data, price_data)
    
    print('计算大额流入/流出指标...')
    large_flow = sentiment_builder.calculate_large_flow(flow_data)
    
    print('计算反常态流入/流出指标...')
    abnormal_flow = sentiment_builder.calculate_abnormal_flow(flow_data)
    
    print('计算背离指标...')
    divergence = sentiment_builder.calculate_divergence_flow(flow_data, price_data)
    
    # 构建情绪指数
    event_data = {
        'counter_market_signal': counter_market,
        'large_flow_signal': large_flow,
        'abnormal_flow_signal': abnormal_flow,
        'divergence_signal': divergence,
    }
    
    selected_events = [
        'counter_market_signal',
        'large_flow_signal',
        'abnormal_flow_signal',
        'divergence_signal',
    ]
    
    sentiment_index = sentiment_builder.build_sentiment_index(
        event_data=event_data,
        selected_events=selected_events
    )
    
    print(f'情绪指数构建完成，数据量: {len(sentiment_index)}')
    display(sentiment_index.head(10))

## 4. 行业配置策略

In [ ]:
print('正在构建行业配置策略...')

# 准备行业收益数据
industries = ['电子', '食品饮料', '银行', '医药生物', '非银金融', 
              '房地产', '汽车', '化工', '家用电器', '计算机']

# 生成模拟的行业收益数据
dates = pd.bdate_range(start=BACKTEST_START_DATE, end=BACKTEST_END_DATE).strftime('%Y%m%d')
np.random.seed(42)

industry_returns_list = []
for date in dates:
    for ind in industries:
        industry_returns_list.append({
            'trade_date': date,
            'industry_code': ind,
            'return': np.random.randn() * 0.02
        })

industry_returns = pd.DataFrame(industry_returns_list)
print(f'行业收益数据构建完成，共 {len(dates)} 个日期，{len(industries)} 个行业')

In [ ]:
# 构建复合因子
print('构建复合因子...')

composite_builder = CompositeFactorStrategy(n_industries=3, frequency='weekly')

# 准备因子数据
factor_data_dict = {}

if 'flow' in northbound_data and not northbound_data['flow'].empty:
    flow_data = northbound_data['flow'].copy()
    
    # 为每个行业生成模拟因子
    for ind in industries:
        ind_factor_data = pd.DataFrame({
            'trade_date': dates,
            'industry_code': ind,
            'factor': np.random.randn(len(dates)) + 0.5
        })
        factor_data_dict[ind] = ind_factor_data

# 合并为单个DataFrame
composite_factor = pd.concat(factor_data_dict.values(), ignore_index=True)

print(f'复合因子构建完成，共 {len(composite_factor)} 条记录')

## 5. 回测与结果

In [ ]:
print('运行回测...')

# 运行行业配置策略
strategy_result = run_allocation_strategy(
    factor_data=composite_factor,
    industry_returns=industry_returns,
    benchmark_returns=benchmark_returns if 'benchmark' in market_data else pd.Series(),
    n_industries=3
)

print('\n=== 策略业绩指标 ===')
metrics = strategy_result['metrics']
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'{k}: {v:.4f}')
    else:
        print(f'{k}: {v}')

In [ ]:
# 分层回测
print('运行分层回测...')

layer_strategy = LayerBacktestStrategy(n_layers=5)
layer_results = layer_strategy.run_layer_backtest(
    factor_data=composite_factor,
    industry_returns=industry_returns
)

print('\n=== 分层回测结果 ===')
for layer_name, result in layer_results.items():
    metrics = result['metrics']
    print(f'\n{layer_name}:')
    print(f'  总收益: {metrics.get("total_return", 0):.4f}')
    print(f'  年化收益: {metrics.get("annualized_return", 0):.4f}')

## 6. 可视化

In [ ]:
print('生成可视化图表...')

# 获取策略收益
if 'returns' in strategy_result and strategy_result['returns'] is not None:
    strategy_returns = strategy_result['returns'].dropna()
    
    if len(strategy_returns) > 0:
        cumulative = (1 + strategy_returns).cumprod() - 1
        
        # 创建图形
        fig, axes = plt.subplots(2, 1, figsize=(14, 10))
        
        # 累计收益曲线
        ax1 = axes[0]
        ax1.plot(cumulative.index, cumulative.values * 100, 'b-', linewidth=2, label='Strategy')
        
        if 'benchmark' in market_data and not market_data['benchmark'].empty:
            benchmark_cum = (1 + benchmark_returns.reindex(strategy_returns.index).fillna(0)).cumprod() - 1
            ax1.plot(cumulative.index, benchmark_cum.values * 100, 'r--', linewidth=1.5, label='HS300 Benchmark')
        
        ax1.set_title('Strategy vs Benchmark Cumulative Return', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Date')
        ax1.set_ylabel('Cumulative Return (%)')
        ax1.legend(loc='best')
        ax1.grid(True, alpha=0.3)
        plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)
        
        # 回撤曲线
        ax2 = axes[1]
        wealth_index = cumulative + 1
        previous_peaks = wealth_index.cummax()
        drawdowns = (wealth_index - previous_peaks) / previous_peaks * 100
        
        ax2.fill_between(cumulative.index, drawdowns.values, 0, alpha=0.3, color='red')
        ax2.plot(cumulative.index, drawdowns.values, 'r-', linewidth=1)
        ax2.set_title('Strategy Drawdown', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Date')
        ax2.set_ylabel('Drawdown (%)')
        ax2.grid(True, alpha=0.3)
        plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)
        
        plt.tight_layout()
        plt.show()
        
        print('可视化完成！')

In [ ]:
# 分层回测可视化
if layer_results:
    print('生成分层回测可视化...')
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    
    colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728', '#9467bd']
    
    # 累计收益曲线
    ax1 = axes[0]
    for i, (layer_name, result) in enumerate(layer_results.items()):
        cum_return = result['cumulative_return']
        if not cum_return.empty:
            ax1.plot(cum_return.index, cum_return.values * 100, 
                     label=layer_name, linewidth=2, color=colors[i % len(colors)])
    
    ax1.set_title('Layer Backtest Cumulative Returns', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Cumulative Return (%)')
    ax1.legend(loc='best')
    ax1.grid(True, alpha=0.3)
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)
    
    # 年化收益对比
    ax2 = axes[1]
    layer_names = list(layer_results.keys())
    ann_returns = [layer_results[name]['metrics'].get('annualized_return', 0) * 100 
                   for name in layer_names]
    
    bars = ax2.bar(layer_names, ann_returns, color=colors[:len(layer_names)])
    ax2.set_title('Layer Annualized Returns Comparison', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Layer')
    ax2.set_ylabel('Annualized Return (%)')
    ax2.grid(True, alpha=0.3, axis='y')
    
    for bar, val in zip(bars, ann_returns):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1, 
                f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    print('分层回测可视化完成！')

## 7. 总结

### 7.1 研报复现说明

本notebook尝试复现华泰证券《析精剖微：机构拆解看北向资金》研究报告的核心内容：

1. **北向资金数据获取**：通过tushare等API获取北向资金流向和持仓数据

2. **因子构建**：
   - 持仓市值因子：北向资金持仓市值与全市场市值的比例
   - 资金流向因子：北向资金流入流出与行业成交额的比例
   - 主动权重因子：北向资金配置权重与基准权重的偏配
   - 机构打分因子：根据净流入机构数目对行业打分

3. **情绪指数**：基于13项事件指标构建情绪指数

4. **行业配置策略**：基于复合因子进行周频和双周频行业配置

### 7.2 数据限制说明

**重要提示**：研报中使用的按机构类型（外资银行、外资券商、内资银行、内资券商）分类的详细持仓数据需要从港交所获取，这部分数据通过免费API无法完全获取。本代码使用了模拟数据进行演示，实际使用时请替换为真实数据。

### 7.3 下一步建议

1. 获取真实的北向资金持仓明细数据
2. 实现Brinson归因分析
3. 进行更精细的机构分类分析
4. 优化策略参数和交易成本